# Architecture

Use an existing model + your tax knowledge base

User (Web UI) --> Backend API  --> Vector Database (your tax documents) -->LLM (OpenAI / Llama / Mistral) -->Answer about 1120 / 1065 efiling


# Training Data

Your system can ingest:

IRS instructions

1120 filing instructions

1065 filing instructions

E-file schema documentation

XML examples

Tax software workflow

Corporate tax FAQs

Internal documentation

IRS publications



# Data Formats supported:

PDF
Word
CSV
HTML
IRS XML schemas
Tax workflow documentation


# Technology Stack (Simple Version)

Hosted models: GPT-4 / GPT-4o , Claude, Gemini

Self hosted : Llama 3 ,Mixtral,DeepSeek

**For tax AI I recommend:Llama 3 70B or GPT-4**



# Vector Database

Stores embeddings of your tax docs.

Popular: Pinecone, Weaviate ,Qdrant ,Chroma



# Backend

Good options:

Python FastAPI (best)

Node.js

Django


# Frontend

React

Next.js

simple chat UI


# Data Pipeline

Tax PDFs
IRS docs
XML schema docs
Internal tax procedures
        |
Document parser
        |
Text chunking
        |
Embeddings
        |
Vector database











# Backend (Python FastAPI)

# Web Interface

Simple chat UI.

# Security & Compliance

encryption

user authentication

audit logging

PII protection

IRS compliance

# Install Required Libraries

In [26]:
!pip install langchain
!pip install sentence-transformers
!pip install chromadb
!pip install pypdf
!pip install transformers

!pip install -q langchain-text-splitters

!pip install -q langchain langchain-community
!pip install -q sentence-transformers chromadb pypdf transformers

from langchain_community.document_loaders import PyPDFLoader


In [24]:
# check the data base folder path and name

import os
os.listdir("/kaggle/input")

['datasets']

In [20]:
"""# Loading one pdf file 


loader = PyPDFLoader("/kaggle/input/datasets/nallagantisharath/efile-dataset/1120.pdf")

docs = loader.load()

print("Pages loaded:", len(docs))"""

Pages loaded: 34


In [49]:
# loading multiple pdf files 

from langchain_community.document_loaders import PyPDFLoader
import os

folder = "/kaggle/input/datasets/nallagantisharath/efile-dataset"

docs = []

for file in os.listdir(folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        docs.extend(loader.load())

print("Total pages loaded:", len(docs))

Total pages loaded: 443


# Split Documents into Chunks

In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total chunks: 2273


# create embeddings

In [51]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


# Create Vector Database 

In [52]:
from langchain_community.vectorstores import Chroma

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Vector database created")

Vector database created


# Test the Knowledge Search

In [53]:
query = "How do I efile Form 1120?"

results = vectordb.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("-----")

company 
(section 831)
1120-PC
Political organization 
(section 527) 1120-POL
Real estate investment trust (section 
856) 1120-REIT
Regulated investment company 
(section 851) 1120-RIC
S corporation (section 1361) 1120-S
Settlement fund 
(section 468B) 1120-SF
Electronic Filing
Corporations can generally electronically file (e-file) Form 1120, 
related forms, schedules, and attachments; Form 7004 
(automatic extension of time to file); and Forms 940, 941, and 
944 (employment tax returns). If there is a balance due, the 
corporation can authorize an electronic funds withdrawal while 
e-filing. Form 1099 and other information returns can also be 
electronically filed. The option to e-file does not, however, apply 
to certain returns.
-----
company 
(section 831)
1120-PC
Political organization 
(section 527) 1120-POL
Real estate investment trust (section 
856) 1120-REIT
Regulated investment company 
(section 851) 1120-RIC
S corporation (section 1361) 1120-S
Settlement fund 
(section 468B

# Generate an AI Answer

In [54]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

context = "\n".join([r.page_content for r in results])

prompt = f"""
Use the tax documentation below to answer the question.

Context:
{context}

Question: {query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


to this address


In [56]:
def ask_tax_ai(question):

    results = vectordb.similarity_search(question, k=10)

    context = "\n".join([r.page_content for r in results])

    prompt = f"""
Use the tax documentation below to answer the question.

Context:
{context}

Question: {question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [57]:
ask_tax_ai("How do I efile Form 1120?")

'Form 1120/2025'

In [ ]:
while True:

    question = input("Ask a tax question (type exit to stop): ")

    if question.lower() == "exit":
        break

    answer = ask_tax_ai(question)

    print("\nAI:", answer)
    print()

Ask a tax question (type exit to stop):  How can I create/prepare a Short Year 1120 Return?



AI: PeriodReasonCd or shortPeriodReasonCd1120-FInd fields to provide the regulatory citation or reason for the Short Period Return.



Ask a tax question (type exit to stop):  My charitable contributions have been entered correctly but they are not carrying to line 19 of the Form 1120. Why not?



AI: Form 8283, Noncash Charitable Contributions



In [14]:
import os
print(os.listdir("Tax_Assistance2"))

model.save_pretrained("Tax_Assistance2")
tokenizer.save_pretrained("Tax_Assistance2")

FileNotFoundError: [Errno 2] No such file or directory: 'Tax_Assistance2'